# Moduł 07. Python Unit Tests — `unittest.mock` Zaawansowane — v2.0

Autor: Kamil Bartocha

## Rozkład jazdy

- 🔹 1. `call`, `call_args`, kolejność wywołań
- 🔹 2. `patch.dict`, `patch.multiple`, `sentinel`
- 🔹 3. `AsyncMock` i testowanie kodu asynchronicznego

## 1. 🔹 `call`, `call_args`, kolejność wywołań

Obiekt `call` z modułu `unittest.mock` reprezentuje jedno wywołanie
funkcji razem z jej argumentami. Porównujemy go z `call_args_list`,
by zweryfikować dokładną historię wywołań mocka.

Kluczowe atrybuty mocka:

| Atrybut | Opis |
|---|---|
| `call_args` | argumenty ostatniego wywołania |
| `call_args_list` | lista wszystkich wywołań |
| `assert_has_calls(calls)` | weryfikuje podciąg wywołań |
| `assert_has_calls(calls, any_order=True)` | kolejność nieistotna |

Do weryfikacji kolejności wywołań kilku mocków używamy `Mock()` jako
menedżera (manager mock), do którego dołączamy atrybuty:

```python
manager = Mock()
manager.attach_mock(mock_a, 'a')
manager.attach_mock(mock_b, 'b')
# manager.mock_calls zawiera wywołania obu atrybutów w kolejności
```

In [ ]:
from unittest.mock import Mock, call

# call_args i call_args_list
logger = Mock()
logger('INFO', 'start')
logger('WARNING', 'slow query')
logger('ERROR', 'timeout')

print('last call_args:', logger.call_args)
print('all calls:')
for c in logger.call_args_list:
    print(' ', c)

# assert_has_calls - sprawdza podciąg (niekoniecznie cały)
logger.assert_has_calls([
    call('INFO', 'start'),
    call('WARNING', 'slow query'),
])
print('assert_has_calls: OK')

# manager - kolejność wywołań wielu mocków
manager = Mock()
db = Mock()
cache = Mock()
manager.attach_mock(db, 'db')
manager.attach_mock(cache, 'cache')

db.get(1)
cache.set(1, 'result')
db.close()

print('manager.mock_calls:', manager.mock_calls)

---

### 🐍 Ćwiczenia — call i kolejność wywołań

1. Użyj `mock.call_args` i `mock.call_args_list` do inspekcji argumentów
2. Porównaj wywołania przez `assert_has_calls([call(1), call(2)])`
3. *(Trudniejsze)* Sprawdź kolejność wywołań dwóch różnych mocków
   przez `manager`

In [ ]:
from unittest.mock import Mock, call

# Ćwiczenie 1: call_args i call_args_list
# Utwórz mock dla funkcji save(entity, mode).
# Wywołaj: save('user', 'insert'), save('order', 'update'), save('log', 'insert').
# Wypisz call_args (ostatnie wywołanie) i call_args_list.
# Sprawdź, że drugie wywołanie miało argumenty ('order', 'update').

save_mock = Mock()

...

print('ostatnie:', save_mock.call_args)
print('wszystkie:', save_mock.call_args_list)

second_call = save_mock.call_args_list[1]
assert second_call == call(...)
print('OK')

In [ ]:
from unittest.mock import Mock, call

# Ćwiczenie 2: assert_has_calls
# Mock dla pipeline.step(name). Wywołaj: step('load'), step('transform'),
# step('validate'), step('save').
# Użyj assert_has_calls, by sprawdzić że 'transform' poprzedza 'validate'.

pipeline = Mock()

...

pipeline.step.assert_has_calls([
    call(...),
    call(...),
])
print('kolejność OK')
print('call_count:', pipeline.step.call_count)

In [ ]:
from unittest.mock import Mock, call

# Ćwiczenie 3: manager - kolejność wywołań *(Trudniejsze)*
# Funkcja process(data) powinna najpierw wywołać validator.check(data),
# potem formatter.format(data), a na koniec writer.write(...).
# Użyj manager.attach_mock, by zweryfikować pełną kolejność.
# hint: manager.mock_calls powinien zawierać 3 wpisy w odpowiedniej kolejności.

validator = Mock()
formatter = Mock()
writer = Mock()

validator.check.return_value = True
formatter.format.return_value = 'formatted_data'

def process(data):
    if validator.check(data):
        formatted = formatter.format(data)
        writer.write(formatted)

manager = Mock()
manager.attach_mock(validator, 'validator')
manager.attach_mock(formatter, 'formatter')
manager.attach_mock(writer, 'writer')

process('raw_data')

expected = [
    call.validator.check(...),
    call.formatter.format(...),
    call.writer.write(...),
]
manager.assert_has_calls(expected)
print('kolejność zachowana: OK')
print('manager.mock_calls:', manager.mock_calls)

## 2. 🔹 `patch.dict`, `patch.multiple`, `sentinel`

`patch.dict` pozwala tymczasowo modyfikować lub zastępować słowniki
(np. `os.environ`) na czas testu:

```python
with patch.dict(os.environ, {'API_KEY': 'test-key'}):
    result = get_api_key()  # widzi 'test-key'
# poza blokiem os.environ wraca do stanu wyjściowego
```

`patch.multiple` podmienia kilka atrybutów modułu jednocześnie:

```python
with patch.multiple('mymodule', funcA=DEFAULT, funcB=DEFAULT) as mocks:
    mocks['funcA'].return_value = 42
```

`sentinel` dostarcza unikalne obiekty wartownicze przydatne
gdy `None` jest prawidłową wartością zwracaną:

```python
from unittest.mock import sentinel
NOT_SET = sentinel.NOT_SET
mock.find.return_value = sentinel.NOT_SET
assert result is sentinel.NOT_SET  # pewna tożsamość
```

In [ ]:
import os
from unittest.mock import patch, patch as p, DEFAULT, sentinel

# patch.dict - zmienne środowiskowe
def get_database_url():
    host = os.environ.get('DB_HOST', 'localhost')
    port = os.environ.get('DB_PORT', '5432')
    return f'postgresql://{host}:{port}/mydb'

with patch.dict(os.environ, {'DB_HOST': 'test-server', 'DB_PORT': '5433'}):
    url = get_database_url()
    print('patched url:', url)

print('restored url:', get_database_url())

# patch.dict z clear=True - czysci slowanik przed podmiana
original = {'a': 1, 'b': 2}
with patch.dict(original, {'c': 3}, clear=True):
    print('during patch:', original)   # {'c': 3}
print('after patch:', original)         # {'a': 1, 'b': 2}

# sentinel
cache = {}
MISSING = sentinel.MISSING

def get_or_default(key, default=None):
    return cache.get(key, MISSING)

result = get_or_default('x')
print('sentinel result is MISSING:', result is MISSING)  # True
print('sentinel repr:', repr(MISSING))

---

### 🐍 Ćwiczenia — patch.dict, sentinel

4. Podmień klucze w słowniku przez `patch.dict`
5. Użyj `sentinel` jako unikalny obiekt wartowniczy
6. *(Trudniejsze)* Napisz wzorzec spy - opakuj prawdziwy obiekt,
   loguj wywołania

In [ ]:
import os
from unittest.mock import patch
import unittest

# Ćwiczenie 4: patch.dict
# Funkcja get_feature_flags() czyta os.environ i zwraca słownik flag.
# Przetestuj z podmienionymi zmiennymi środowiskowymi.

def get_feature_flags():
    return {
        'dark_mode': os.environ.get('FEATURE_DARK_MODE', 'off') == 'on',
        'beta': os.environ.get('FEATURE_BETA', 'off') == 'on',
    }


class TestFeatureFlags(unittest.TestCase):
    def test_flags_enabled(self):
        with patch.dict(os.environ, {...}):
            flags = get_feature_flags()
            self.assertTrue(flags['dark_mode'])
            self.assertTrue(flags['beta'])
            print('flags:', flags)

    def test_flags_default_off(self):
        with patch.dict(os.environ, {}, clear=True):
            flags = get_feature_flags()
            self.assertFalse(flags['dark_mode'])
            self.assertFalse(flags['beta'])

unittest.main(argv=[''], exit=False, verbosity=2)

In [ ]:
from unittest.mock import Mock, sentinel

# Ćwiczenie 5: sentinel jako wartownik
# Funkcja find_first(collection, predicate) zwraca pierwszy pasujący element
# lub sentinel.NOT_FOUND gdy nic nie pasuje.
# Sentinel jest lepszy niż None, bo None może być prawidłową wartością.
# Zaimplementuj i przetestuj.

def find_first(collection, predicate):
    for item in collection:
        if predicate(item):
            return item
    return ...


result = find_first([1, 2, 3, 4], lambda x: x > 10)
print('not found:', result is sentinel.NOT_FOUND)  # True

result2 = find_first([None, None, 5], lambda x: x is not None)
print('found:', result2)  # 5

In [ ]:
from unittest.mock import Mock

# Ćwiczenie 6: wzorzec spy *(Trudniejsze)*
# Spy opakowuje prawdziwy obiekt - przekazuje wywołania dalej,
# ale rejestruje je do późniejszej weryfikacji.
# Utwórz SpyCalculator opakowujący Calculator i logujący wywołania.

class Calculator:
    def add(self, a, b):
        return a + b

    def multiply(self, a, b):
        return a * b


class SpyCalculator:
    def __init__(self, real):
        self._real = real
        self.calls = []

    def add(self, a, b):
        result = ...
        self.calls.append(('add', a, b, result))
        return result

    def multiply(self, a, b):
        result = ...
        self.calls.append(('multiply', a, b, result))
        return result


spy = SpyCalculator(Calculator())
print('add:', spy.add(3, 4))       # 7
print('multiply:', spy.multiply(3, 4))  # 12
print('calls:', spy.calls)

## 3. 🔹 `AsyncMock` i testowanie kodu asynchronicznego

Od Pythona 3.8 moduł `unittest.mock` dostarcza `AsyncMock` -
atrapę funkcji asynchronicznych (`async def`). Wywołanie `AsyncMock`
zwraca coroutine, który można `await`ować w teście.

Testujemy kod asynchroniczny przez `asyncio.run()` lub przez
narzędzia takie jak `pytest-asyncio`:

```python
from unittest.mock import AsyncMock
import asyncio

async def test_async():
    mock_fetch = AsyncMock(return_value={'id': 1})
    result = await mock_fetch('/api/user/1')
    assert result == {'id': 1}

asyncio.run(test_async())
```

`AsyncMock` obsługuje te same atrybuty co `Mock`:
`return_value`, `side_effect`, `call_count`, `assert_called_with`.

In [ ]:
from unittest.mock import AsyncMock, patch
import asyncio

# Przykład 1: AsyncMock - podstawowe użycie
async def fetch_user(user_id, http_client):
    response = await http_client.get(f'/users/{user_id}')
    return response['name']

async def demo_async_mock():
    mock_client = AsyncMock()
    mock_client.get.return_value = {'name': 'Alice', 'id': 1}

    name = await fetch_user(1, mock_client)
    print('fetched name:', name)

    mock_client.get.assert_called_once_with('/users/1')
    print('assert_called_once_with: OK')

asyncio.run(demo_async_mock())

# Przykład 2: side_effect w AsyncMock
async def demo_side_effect():
    mock_db = AsyncMock()
    mock_db.query.side_effect = [{'id': 1}, {'id': 2}, ConnectionError('brak połączenia')]

    r1 = await mock_db.query('SELECT 1')
    r2 = await mock_db.query('SELECT 2')
    print('r1:', r1, 'r2:', r2)

    try:
        await mock_db.query('SELECT 3')
    except ConnectionError as e:
        print('błąd:', e)

asyncio.run(demo_side_effect())

---

### 🐍 Ćwiczenia — AsyncMock

7. Przetestuj `async def fetch_user(id)` przez `AsyncMock`
8. Przetestuj `async def process_batch(items)` mockując wewnętrzny `await`
9. *(Trudniejsze)* Przetestuj serwis zależny od dwóch mockowanych klientów API

In [ ]:
from unittest.mock import AsyncMock
import asyncio

# Ćwiczenie 7: AsyncMock - fetch_user
# Funkcja fetch_user(user_id, repo) wywołuje await repo.find(user_id)
# i zwraca user['email'].
# Przetestuj przez AsyncMock z return_value = {'email': 'alice@example.com'}.

async def fetch_user(user_id, repo):
    user = await repo.find(user_id)
    return user['email']


async def test_fetch_user():
    mock_repo = AsyncMock()
    mock_repo.find.return_value = ...

    email = await fetch_user(42, mock_repo)
    assert email == 'alice@example.com'
    mock_repo.find.assert_called_once_with(...)
    print('email:', email)

asyncio.run(test_fetch_user())

In [ ]:
from unittest.mock import AsyncMock, patch
import asyncio

# Ćwiczenie 8: process_batch z wewnętrznym await
# Funkcja process_batch(items) iteruje po liście i dla każdego elementu
# wywołuje await save_item(item). Podmień save_item przez AsyncMock
# i sprawdź call_count oraz że wszystkie elementy zostały przetworzone.

async def save_item(item):
    raise RuntimeError('prawdziwy zapis - nie wywoływać!')

async def process_batch(items):
    for item in items:
        await save_item(item)
    return len(items)


async def test_process_batch():
    mock_save = AsyncMock(return_value=None)
    with patch('__main__.save_item', mock_save):
        count = await process_batch([...])  # podaj 3 elementy

    assert count == 3
    assert mock_save.call_count == ...
    print('processed:', count, 'call_count:', mock_save.call_count)

asyncio.run(test_process_batch())

In [ ]:
from unittest.mock import AsyncMock
import asyncio

# Ćwiczenie 9: serwis z dwoma klientami API *(Trudniejsze)*
# Klasa OrderService.create_order(user_id, product_id) wykonuje:
#   1. user = await user_client.get_user(user_id)
#   2. product = await product_client.get_product(product_id)
#   3. zwraca {'user': user['name'], 'product': product['title'], 'total': product['price']}
# Przetestuj przez dwa AsyncMock z odpowiednimi return_value.

class OrderService:
    def __init__(self, user_client, product_client):
        self.user_client = user_client
        self.product_client = product_client

    async def create_order(self, user_id, product_id):
        user = await self.user_client.get_user(user_id)
        product = await self.product_client.get_product(product_id)
        return {
            'user': user['name'],
            'product': product['title'],
            'total': product['price'],
        }


async def test_create_order():
    user_client = AsyncMock()
    user_client.get_user.return_value = ...

    product_client = AsyncMock()
    product_client.get_product.return_value = ...

    service = OrderService(user_client, product_client)
    order = await service.create_order(1, 101)

    assert order['user'] == ...
    assert order['product'] == ...
    print('order:', order)

asyncio.run(test_create_order())